# Render Incipits (or any measure range) with Verovio

## Load libraries

In [1]:
import IPython
from IPython.display import HTML, Javascript, display, SVG
# import crim_intervals and specific modules
import crim_intervals
from crim_intervals import * 
from crim_intervals import main_objs
import os
import glob as glob
import verovio
import bs4
from bs4 import BeautifulSoup
from pathlib import Path

print('Ok')

Ok


In [2]:
def verovioPrintExample2(piece, start, stop):

        """
        Pass a range of measures (as integers) to print the given range.

        For last measure you can also use '-1', thus for all measures:

        verovioPrintExample(1, -1)

        """
        if piece.path.startswith('Music_Files/'):
            # Convert the relative path to an absolute path
            absolute_path = os.path.abspath(piece.path)
            text_file = open(absolute_path, "r")
            fetched_mei_string = text_file.read()
        elif piece.path.startswith('/'):
            # If the path is already absolute, no need to convert
            text_file = open(piece.path, "r")
            fetched_mei_string = text_file.read()
        else:
            response = httpx.get(piece.path)
            fetched_mei_string = response.text
        tk = verovio.toolkit()
        tk.loadData(fetched_mei_string)
        tk.setScale(30)
        tk.setOptions({"pageHeight":  1500, # Height in pixels
                       "pageWidth":  3000    # Width in pixels
                       })

        if stop == -1:
            meas = self.measures()
            stop = meas.iloc[-1].tolist()[0]

        mr = str(start) + "-" + str(stop)
        mdict = {'measureRange': mr}

        if stop < start:
            print("Check the measure range, the stop measure must be equal to or greater than the start measure")
        else:# select verovio measures and redo layout
            # tk.select(str(mdict))
            tk.select(mdict)
            tk.redoLayout()

            # get the number of pages and display the music
            print("Score:")
            count = tk.getPageCount()
            for c in range(1, count + 1):
                music = tk.renderToSVG(c)
                # print("File Name: ", selfpiecefile_name)
                print(piece.metadata['composer'])
                print(piece.metadata['title'])
                print("Measures: " + str(start) + "-" + str(stop))
                display(HTML(music))

### Check Corpus

In [4]:
corpus_list = sorted(glob.glob('Music_Files/*'))
corpus_list

[]

## Function to Print XML with Verovio

In [16]:
def verovioPrint(xml, soup, file_name, file_extension):
        tk = verovio.toolkit()
        tk.loadData(xml)
        tk.setScale(50)
        tk.setOptions({"pageHeight":  1500, # Height in pixels
                       "pageWidth":  1500    # Width in pixels
                       })
        count = tk.getPageCount()
        for c in range(1, count + 1):
            music = tk.renderToSVG(c)
            print("File_Name: " + file_name)
            # print("Composer: " + soup.find('persName', {'role' : 'composer'}).text.strip())
            if file_extension == '.mei': 
                if soup.find('persName', {'role' : 'composer'}).text.strip() is not None:
                    print("Composer: " + soup.find('persName', {'role' : 'composer'}).text.strip())
                if soup.find('title').text.strip() is not None:
                    print("Title: " + soup.find('title').text.strip())
            elif file_extension == '.musicxml': 
                if soup.find('creator', attrs={'type': 'composer'}).text.strip() is not None:
                    print("Composer: " + soup.find('creator', attrs={'type': 'composer'}).text.strip())
                if soup.find('work').find('work-title').text.strip() is not None:
                    print("Title: " + soup.find('work').find('work-title').text.strip())
            print("Measures: " + str(measure_range[0]) + "-" + str(measure_range[1]))
            display(HTML(music))
            

            

## Take in any folder of files and Render measure range with verovio

In [17]:
# %%capture --no-display

# the %%capture code must be the FIRST LINE in order to suppress warnings!

# Set measure range
measure_range = [1, 2]

# Specify the name of the directory you want to create 
file_directory = "Updates/*.musicxml"

# get the corpus
corpus_list = sorted(glob.glob(file_directory))

# process files for path, name
for item in corpus_list:
    # Convert the path to an absolute path if needed
    item_path = os.path.abspath(item)
    
    # Extract the stem and extension of the original file
    path_obj = Path(item_path)
    
    # Use .name .stem and .suffix on the Path object
    file_name = path_obj.name
    file_stem = path_obj.stem
    file_extension = path_obj.suffix

    # get the xml
    with open(item_path, 'r') as xml_file:
        xml_data = xml_file.read() 
        
    # Parse the XML data with BeautifulSoup
        soup = BeautifulSoup(xml_data, 'lxml-xml')
        
        if file_extension == '.mei':
            # Find all <measure> elements and remove those not matching the specified "n" values
            for measure in soup.find_all('measure'):
                n_value = measure.get('n')
                if n_value is not None and int(n_value) not in measure_range:
                    measure.decompose() # Remove the element

        if file_extension == '.musicxml':
            # Find all <measure> elements and remove those not matching the specified "n" values
            for measure in soup.find_all('measure'):
                n_value = measure.get('number')
                if n_value is not None and int(n_value) not in measure_range:
                    measure.decompose() # Remove the element
        xml_string = str(soup)

        # Now, pass the sliced XML string to the VerovioPrint function
        verovioPrint(xml_string, soup, file_name, file_extension)
        
    


File_Name: Riccio_1_01_Exultate_Deo_2vv.musicxml
Composer: Giovanni Battista Riccio
Title: 1_01_Exultate Deo
Measures: 1-2


File_Name: Riccio_1_02_Misericordias_Domini_2vv.musicxml
Composer: Giovanni Battista Riccio
Title: 1_02_Misericordias Domini
Measures: 1-2


File_Name: Riccio_1_03_Audite_omnes_gentes_2vv.musicxml
Composer: Giovanni Battista Riccio
Title: 1_03_Audite omnes gentes
Measures: 1-2


File_Name: Riccio_1_04_O_bone_Iesu_2vv.musicxml
Composer: Giovanni Battista Riccio
Title: 1_04_O bone Iesu
Measures: 1-2


File_Name: Riccio_1_05_Cantate_Domino_2vv.musicxml
Composer: Giovanni Battista Riccio
Title: 1_05_Cantate Domino
Measures: 1-2


...

File_Name: Riccio_1_31_O_magnum_misterium_2vv.musicxml
Composer: Giovanni Battista Riccio
Title: 1_31_O magnum mysterium
Measures: 1-2


File_Name: Riccio_1_32_O_dulcissime_a_tre_3vv.musicxml
Composer: Giovanni Battista Riccio
Title: 1_32_O dulcissime
Measures: 1-2


File_Name: Riccio_1_33_O_quam_gloriosum_1v.musicxml
Composer: Giovanni Battista Riccio
Title: 1_33_O quam gloriosum
Measures: 1-2


File_Name: Riccio_1_34_Canzon_[I]_2vv.musicxml
Composer: Giovanni Battista Riccio
Title: 1_34_Canzon [I]
Measures: 1-2


File_Name: Riccio_1_35_Canzon_a_doi_flautini_[I]_2vv.musicxml
Composer: Giovanni Battista Riccio
Title: 1_35_Canzon [II]
Measures: 1-2
